### Importation des bibliothèques

In [1]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import scipy.sparse as sp, scipy.io, os, subprocess
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
import os
import re
import numpy as np
import time
import pandas as pd

### Configuration des chemins et des dossiers de travail

In [2]:
BASE_DIR = r"C:\Users\Home\Documents\M2\projet_PPD"
DATA_ROOT = f"{BASE_DIR}\data"
KMEANS_ROOT = f"{BASE_DIR}\output\KMeans"
PALMETTO_JAR = f"{BASE_DIR}\palmetto\palmetto-0.1.0-jar-with-dependencies.jar"
WIKI_BD_DIR = f"{BASE_DIR}\palmetto\Wikipedia_bd\wikipedia_bd"

os.makedirs(KMEANS_ROOT, exist_ok=True)
print(f"Data: {DATA_ROOT}")
print(f"KMeans: {KMEANS_ROOT}")
print(f"Palmetto: {PALMETTO_JAR}")


Data: C:\Users\Home\Documents\M2\projet_PPD\data
KMeans: C:\Users\Home\Documents\M2\projet_PPD\output\KMeans
Palmetto: C:\Users\Home\Documents\M2\projet_PPD\palmetto\palmetto-0.1.0-jar-with-dependencies.jar


### Initialisation du seed et chargement des datasets

In [3]:
SEED = 42
def set_seed(seed): import random; random.seed(seed); np.random.seed(seed)

def load_dataset(name):
    path = os.path.join(DATA_ROOT, name)
    train = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()
    vocab = [w.strip() for w in open(os.path.join(path, "vocab.txt"), encoding="utf-8")]
    labels = np.loadtxt(os.path.join(path, "train_labels.txt"), dtype=int)
    return train, test, vocab, labels

set_seed(SEED); print("Fonctions prêtes")


Fonctions prêtes


### Fonctions pour les métriques d'évaluation

In [4]:
def topic_diversity(topics): 
    words = [w for t in topics for w in t]
    return len(set(words)) / len(words) if words else 0.0

def purity_score(y_true, y_pred):
    total = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        labels, counts = np.unique(y_true[idx], return_counts=True)
        total += counts.max()
    return total / len(y_true)

def compute_nmi(theta, labels):
    pred = theta.argmax(axis=1)
    return normalized_mutual_info_score(labels, pred)


### Calcul des métriques d'évaluation pour K-Means

In [5]:
def compute_metrics_kmeans(dataset, K, seed=42, topn=15):
    dataset_dir = f"{KMEANS_ROOT}/{dataset}"
    mat_path = f"{dataset_dir}/{dataset}_KMeans_K{K}_seed{seed}.mat"
    
    if not os.path.isfile(mat_path):
        print(f".mat manquant")
        return None
    
    loaded = scipy.io.loadmat(mat_path)
    beta = loaded["beta"]
    train_theta = loaded["train_theta"]
    
    _, _, vocab, labels = load_dataset(dataset)
    
    # TD (Topic Diversity)
    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topics.append([vocab[i] for i in top_idx])
    td = topic_diversity(topics)
    
    # Purity et NMI
    y_pred = train_theta.argmax(axis=1)
    pur = purity_score(labels, y_pred)
    nmi = compute_nmi(train_theta, labels)
    
    # Sauvegarde
    out_path = f"{dataset_dir}/metrics_{dataset}_KMeans_K{K}_seed{seed}.txt"
    with open(out_path, "w") as f:
        f.write(f"dataset={dataset}\nmodel=KMeans\nK={K}\nseed={seed}\n")
        f.write(f"TD={td:.6f}\nPurity={pur:.6f}\nNMI={nmi:.6f}\n")
    
    print(f"Metrics OK: TD={td:.4f} | Purity={pur:.4f} | NMI={nmi:.4f}")
    return out_path


### Calcul de la cohérence C_V avec Palmetto

In [6]:
def compute_palmetto_kmeans(dataset, K, seed=42, topn=15):
    dataset_dir = f"{KMEANS_ROOT}/{dataset}"
    mat_path = f"{dataset_dir}/{dataset}_KMeans_K{K}_seed{seed}.mat"
    
    if not os.path.isfile(mat_path):
        print(f".mat manquant: {mat_path}")
        return None
    
    loaded = scipy.io.loadmat(mat_path)
    beta = loaded["beta"]
    _, _, vocab = load_dataset(dataset)[:3]
    
    topics_file = f"{dataset_dir}/topics_for_cv_{dataset}_KMeans_K{K}_seed{seed}.txt"
    cv_file = f"{dataset_dir}/palmetto_CV_{dataset}_KMeans_K{K}_seed{seed}.txt"
    
    with open(topics_file, "w", encoding="utf-8") as f:
        for k in range(beta.shape[0]):
            top_idx = np.argsort(beta[k])[::-1][:topn]
            f.write(" ".join(vocab[i] for i in top_idx) + "\n")
    
    cmd = ["java", "-jar", PALMETTO_JAR, WIKI_BD_DIR, "C_V", topics_file]
    try:
        out = subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)
        with open(cv_file, "w", encoding="utf-8") as f:
            f.write(out)
        print(f"Palmetto OK: {cv_file}")
        return cv_file
    except:
        print(f"Erreur Palmetto")
        return None


### Entraînement du modèle K-Means 

In [7]:
def train_one_kmeans(dataset, K, seed=42):
    set_seed(seed)
    dataset_dir = f"{KMEANS_ROOT}/{dataset}"
    os.makedirs(dataset_dir, exist_ok=True)
    mat_path = f"{dataset_dir}/{dataset}_KMeans_K{K}_seed{seed}.mat"
    
    if os.path.exists(mat_path): return mat_path
    
    print(f"KMeans | {dataset} | K={K}")
    train_bow, test_bow, vocab, train_labels = load_dataset(dataset)
    
    from sklearn.preprocessing import normalize
    train_bow_norm = normalize(train_bow)
    test_bow_norm = normalize(test_bow)
    
    
    kmeans = KMeans(
    n_clusters=K,
    init="k-means++",
    random_state=42,
    n_init=20,
    max_iter=100,
    tol=1e-4)


    train_clusters = kmeans.fit_predict(train_bow_norm)
    test_clusters = kmeans.predict(test_bow_norm)
    
    
    beta = kmeans.cluster_centers_
    beta = beta / (beta.sum(axis=1, keepdims=True) + 1e-10)
    
    # Theta one-hot (reste identique)
    train_theta = np.zeros((len(train_clusters), K))
    train_theta[np.arange(len(train_clusters)), train_clusters] = 1.0
    test_theta = np.zeros((len(test_clusters), K))
    test_theta[np.arange(len(test_clusters)), test_clusters] = 1.0
    
    scipy.io.savemat(mat_path, {
        "beta": beta.astype(np.float32),
        "train_theta": train_theta.astype(np.float32),
        "test_theta": test_theta.astype(np.float32)
    })
    print(f"Sauvegardé: {mat_path}")
    return mat_path


### Batch d'entraînement K-Means sur 4 datasets

In [8]:
print("=" * 60)
print("ENTRAINEMENT KMEANS - 4 BASES")
print("=" * 60)

DATASETS = ['20NG', 'YahooAnswer', 'AGNews', 'IMDB']
K_VALUES = [20,50, 100]
total_processed = total_skipped = total_models = 0

for dataset in DATASETS:
    print(f"\n{'='*50}")
    print(f"BASE: {dataset}")
    
    for K in K_VALUES:
        total_models += 1  # ← Compteur total
        dataset_dir = f"{KMEANS_ROOT}/{dataset}"
        mat_path = f"{dataset_dir}/{dataset}_KMeans_K{K}_seed{SEED}.mat"
        
        print(f"\n--- {dataset} K={K} ---")
        
        if os.path.exists(mat_path):
            print(f"SKIP: {os.path.basename(mat_path)}")
            total_skipped += 1
            continue
        
        print("Calcul KMeans...")
        mat_path = train_one_kmeans(dataset, K, seed=SEED)
        total_processed += 1
        print(f"✓ {dataset} K={K} OK")

print(f"\n{'='*60}")
print(f"Traitées: {total_processed} | Sautees: {total_skipped} | Total: {total_models}")


ENTRAINEMENT KMEANS - 4 BASES

BASE: 20NG

--- 20NG K=20 ---
SKIP: 20NG_KMeans_K20_seed42.mat

--- 20NG K=50 ---
SKIP: 20NG_KMeans_K50_seed42.mat

--- 20NG K=100 ---
SKIP: 20NG_KMeans_K100_seed42.mat

BASE: YahooAnswer

--- YahooAnswer K=20 ---
SKIP: YahooAnswer_KMeans_K20_seed42.mat

--- YahooAnswer K=50 ---
SKIP: YahooAnswer_KMeans_K50_seed42.mat

--- YahooAnswer K=100 ---
SKIP: YahooAnswer_KMeans_K100_seed42.mat

BASE: AGNews

--- AGNews K=20 ---
SKIP: AGNews_KMeans_K20_seed42.mat

--- AGNews K=50 ---
SKIP: AGNews_KMeans_K50_seed42.mat

--- AGNews K=100 ---
SKIP: AGNews_KMeans_K100_seed42.mat

BASE: IMDB

--- IMDB K=20 ---
SKIP: IMDB_KMeans_K20_seed42.mat

--- IMDB K=50 ---
SKIP: IMDB_KMeans_K50_seed42.mat

--- IMDB K=100 ---
SKIP: IMDB_KMeans_K100_seed42.mat

Traitées: 0 | Sautees: 12 | Total: 12


### Calcul des métriques K-Means (TD + Purity + NMI)

In [9]:
def read_palmetto_cv(dataset, K, seed=42):
    cv_path = os.path.join(
        KMEANS_ROOT,
        dataset,
        f"palmetto_CV_{dataset}_KMeans_K{K}_seed{seed}.txt"
    )

    if not os.path.exists(cv_path):
        print("C_V = NOT FOUND")
        return None

    scores = []
    pattern = re.compile(r"\b(\d+[,\.]\d+)\b")

    with open(cv_path, "r", errors="ignore") as f:
        for line in f:
            if "INFO" in line:
                continue

            match = pattern.search(line)
            if match:
                try:
                    score = float(match.group(1).replace(",", "."))
                    scores.append(score)
                except ValueError:
                    pass

    if not scores:
        print("C_V = PARSE ERROR")
        return None

    cv = float(np.mean(scores))
    print(f"C_V = {cv:.4f}")
    return cv


In [10]:
print("=" * 60)
print("METRICS KMEANS (CV PALMETTO + TD + PURITY + NMI)")
print("=" * 60)

for dataset in DATASETS:
    print(f"\n{'='*50}")
    print(f"BASE: {dataset}")
    
    for K in K_VALUES:
        print(f"\n--- {dataset} K={K} ---")
        
        # Chemin attendu du fichier de résultat Palmetto
        # (On utilise la même logique que dans vos fonctions)
        cv_file = os.path.join(KMEANS_ROOT, dataset, f"palmetto_CV_{dataset}_KMeans_K{K}_seed{SEED}.txt")

        # 1. Verification de l'existence
        if os.path.exists(cv_file):
            print(f"Fichier de coherence deja present : {cv_file}")
            print("Saut du calcul Palmetto.")
        else:
            # 2. Lancement du calcul seulement si le fichier n'existe pas
            print("Fichier manquant. Lancement du calcul Palmetto (Java)...")
            compute_palmetto_kmeans(dataset, K, seed=SEED)
        
        # 3. Lecture du score (que le fichier soit vieux ou nouveau)
        read_palmetto_cv(dataset, K, SEED)
        
        # 4. Calcul des autres métriques (TD, Purity, NMI)
        compute_metrics_kmeans(dataset, K)

METRICS KMEANS (CV PALMETTO + TD + PURITY + NMI)

BASE: 20NG

--- 20NG K=20 ---
Fichier de coherence deja present : C:\Users\Home\Documents\M2\projet_PPD\output\KMeans\20NG\palmetto_CV_20NG_KMeans_K20_seed42.txt
Saut du calcul Palmetto.
C_V = 0.3867
Metrics OK: TD=0.3800 | Purity=0.2637 | NMI=0.2399

--- 20NG K=50 ---
Fichier de coherence deja present : C:\Users\Home\Documents\M2\projet_PPD\output\KMeans\20NG\palmetto_CV_20NG_KMeans_K50_seed42.txt
Saut du calcul Palmetto.
C_V = 0.3879
Metrics OK: TD=0.3160 | Purity=0.3779 | NMI=0.3097

--- 20NG K=100 ---
Fichier de coherence deja present : C:\Users\Home\Documents\M2\projet_PPD\output\KMeans\20NG\palmetto_CV_20NG_KMeans_K100_seed42.txt
Saut du calcul Palmetto.
C_V = 0.3887
Metrics OK: TD=0.2967 | Purity=0.4011 | NMI=0.3164

BASE: YahooAnswer

--- YahooAnswer K=20 ---
Fichier de coherence deja present : C:\Users\Home\Documents\M2\projet_PPD\output\KMeans\YahooAnswer\palmetto_CV_YahooAnswer_KMeans_K20_seed42.txt
Saut du calcul Palmetto.
C

### Analyse comparative : Résultats originaux vs Reproduction

In [12]:
data_papier = {
    'Dataset': ['20NG', '20NG', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews', 'IMDB', 'IMDB'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_Papier': [0.251, 0.294, 0.213, 0.244, 0.271,0.297, 0.241, 0.289],
    'TD_Papier': [0.204, 0.317, 0.219, 0.302, 0.242, 0.345, 0.264, 0.395]
}


data_repro = {
    'Dataset': ['20NG', '20NG', 'YahooAnswer', 'YahooAnswer', 'AGNews', 'AGNews', 'IMDB', 'IMDB'],
    'K': [50, 100, 50, 100, 50, 100, 50, 100],
    'CV_Repro': [0.388, 0.389, 0.329, 0.326, 0.389, 0.393, 0.336, 0.334],
    'TD_Repro': [0.316, 0.297, 0.305, 0.279, 0.500, 0.453, 0.105, 0.120],
    'Purity_R': [0.378, 0.401, 0.284, 0.322, 0.573, 0.648, 0.614, 0.624],
    'NMI_R':    [0.310, 0.316, 0.100, 0.120, 0.177, 0.192, 0.023, 0.023]
}

df_p = pd.DataFrame(data_papier)
df_r = pd.DataFrame(data_repro)
df_final = pd.merge(df_p, df_r, on=['Dataset', 'K'])

# Calcul des écarts (Repro - Papier)
df_final['Diff_CV'] = (df_final['CV_Repro'] - df_final['CV_Papier']).round(3)
df_final['Diff_TD'] = (df_final['TD_Repro'] - df_final['TD_Papier']).round(3)


colonnes = [
    'Dataset', 'K', 
    'CV_Repro', 'TD_Repro', 'Purity_R', 'NMI_R', # REPRODUCTION
    'CV_Papier', 'TD_Papier',                    # PAPIER
    'Diff_CV', 'Diff_TD'                         # ECARTS
]

print("="*125)
print(f"{'NOTRE REPRODUCTION':>55} | {'PAPIER ORIGINAL':>22} | {'ECARTS':>18}")
print("="*125)
print(df_final[colonnes].to_string(index=False))

                                     NOTRE REPRODUCTION |        PAPIER ORIGINAL |             ECARTS
    Dataset   K  CV_Repro  TD_Repro  Purity_R  NMI_R  CV_Papier  TD_Papier  Diff_CV  Diff_TD
       20NG  50     0.388     0.316     0.378  0.310      0.251      0.204    0.137    0.112
       20NG 100     0.389     0.297     0.401  0.316      0.294      0.317    0.095   -0.020
YahooAnswer  50     0.329     0.305     0.284  0.100      0.213      0.219    0.116    0.086
YahooAnswer 100     0.326     0.279     0.322  0.120      0.244      0.302    0.082   -0.023
     AGNews  50     0.389     0.500     0.573  0.177      0.271      0.242    0.118    0.258
     AGNews 100     0.393     0.453     0.648  0.192      0.297      0.345    0.096    0.108
       IMDB  50     0.336     0.105     0.614  0.023      0.241      0.264    0.095   -0.159
       IMDB 100     0.334     0.120     0.624  0.023      0.289      0.395    0.045   -0.275
